# Comparacion de modelos de clasificacion HistGradientBoosting, LightGBM, CatBoost, XGBoost

In [1]:
# Standard libraries
import os
import warnings

In [2]:

# Data manipulation and numerical computation
import numpy as np
import pandas as pd


In [3]:
# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

In [4]:
# Scikit-learn utilities
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, RandomizedSearchCV, cross_val_predict
from sklearn.metrics import (
    balanced_accuracy_score, f1_score, confusion_matrix, classification_report
)
from sklearn.calibration import CalibratedClassifierCV

In [5]:

from typing import Dict, Any, List, Optional, Tuple
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.stats import loguniform, randint, uniform
from sklearn.base import BaseEstimator, ClassifierMixin, clone
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.model_selection import StratifiedKFold, RandomizedSearchCV, cross_val_predict
from sklearn.metrics import balanced_accuracy_score, f1_score, confusion_matrix, classification_report
from sklearn.calibration import CalibratedClassifierCV
from sklearn.ensemble import HistGradientBoostingClassifier

# --- add near imports ---
import os, json, re
from pathlib import Path
from datetime import datetime
import joblib

import os, json
from pathlib import Path
from datetime import datetime
import joblib
import re



In [6]:
# Statistical distributions
from scipy.stats import loguniform, randint, uniform

# Typing utilities
from typing import Dict, Any, List, Optional, Tuple

In [7]:
sns.set(style="ticks", context="notebook", palette="deep")
pd.set_option('display.max_columns', None)
palette = {'Bad':'#b2182b','Poor':'#d6604d','Moderate':'#f1a340','Good':'#5aae61','High':'#1b7837'}

In [8]:
path = "../../../data/processed/"
dfs = {}

# read 03_CLEAN_COMPLETE_DF.parquet
df1 = pd.read_parquet(os.path.join(path, "taxones_pressure_epm_train.parquet"))
df2 = pd.read_parquet(os.path.join(path, "taxones_pressure_epm_predict.parquet"))
df = pd.concat([df1, df2], ignore_index=True)

In [9]:
# change unassessed values to NaN
df = df.replace("Unassessed", np.nan)
df = df.replace("None", np.nan)

# index SamplingOperations_code
df = df.set_index('SamplingOperations_code')
 
# DROP HERlvl2Code	Altitude Longitude_Lambert93	Latitude_Lambert93	Watershed	CodeDepartement	HERlvl1Code
df = df.drop(columns=['HERlvl2Code', 'HERlvl2Name', 'HERlvl1Name', 'Altitude','Longitude_Lambert93','Latitude_Lambert93','Watershed','CodeDepartement', 'Date_SamplingOperation'])
df

CodeSite_SamplingOperations_x  \
SamplingOperations_code                                 
S02000008_20170703                          S02000008   
S02000008_20200708                          S02000008   
S02000010_20070906                          S02000010   
S02000010_20090721                          S02000010   
S02000010_20110723                          S02000010   
...                                               ...   
S06940940_20100708                          S06940940   
S06940940_20230623                          S06940940   
S06960950_20160629                          S06960950   
S06960950_20180719                          S06960950   
S06970900_20150601                          S06970900   

                         TotalAbundance_SamplingOperation  Achaa01  Achac01  \
SamplingOperations_code                                                       
S02000008_20170703                                    405      NaN      NaN   
S02000008_20200708                                    400      NaN      NaN   
S02000010_20070906                                    400      NaN      NaN   
S02000010_20090721                                    400      NaN      NaN   
S02000010_20110723                                    398      NaN      NaN   
...                                                   ...      ...      ...   
S06940940_20100708                                    438      NaN      NaN   
S06940940_20230623                                    408      NaN      NaN   
S06960950_20160629                                    401      NaN      NaN   
S06960950_20180719                                    416      NaN      NaN   
S06970900_20150601                                    432      NaN      NaN   

                         Achaf01    Achaf02  Achal01  Acham01  Achan01  \
SamplingOperations_code                                                  
S02000008_20170703           NaN        NaN      NaN      NaN      NaN   
S02000008_20200708           NaN        NaN      NaN      NaN      NaN   
S02000010_20070906           NaN        NaN      NaN      NaN      NaN   
S02000010_20090721           NaN        NaN      NaN      NaN      NaN   
S02000010_20110723           NaN        NaN      NaN      NaN      NaN   
...                          ...        ...      ...      ...      ...   
S06940940_20100708           NaN        NaN      NaN      NaN      NaN   
S06940940_20230623           NaN        NaN      NaN      NaN      NaN   
S06960950_20160629           NaN        NaN      NaN      NaN      NaN   
S06960950_20180719           NaN  19.230769      NaN      NaN      NaN   
S06970900_20150601           NaN        NaN      NaN      NaN      NaN   

                         Achat01   Achat02  Achat03  Achba01  Achbi01  \
SamplingOperations_code                                                 
S02000008_20170703           NaN       NaN      NaN      NaN      NaN   
S02000008_20200708           NaN       NaN      NaN      NaN      NaN   
S02000010_20070906           NaN       NaN      NaN      NaN      NaN   
S02000010_20090721           NaN       NaN      NaN      NaN      NaN   
S02000010_20110723           NaN       NaN      NaN      NaN      NaN   
...                          ...       ...      ...      ...      ...   
S06940940_20100708           NaN       NaN      NaN      NaN      NaN   
S06940940_20230623           NaN       NaN      NaN      NaN      NaN   
S06960950_20160629           NaN       NaN      NaN      NaN      NaN   
S06960950_20180719           NaN       NaN      NaN      NaN      NaN   
S06970900_20150601           NaN  6.944444      NaN      NaN      NaN   

                         Achbi02   Achbr01  Achca01  Achca02  Achca03  \
SamplingOperations_code                                                 
S02000008_20170703           NaN       NaN      NaN      NaN      NaN   
S02000008_20200708           NaN       NaN      NaN      NaN      NaN   
S02000010_20070906           NaN       NaN      N

In [10]:
# keep the numerical columns only
df = df.select_dtypes(include=[np.number, 'category'])
df

TotalAbundance_SamplingOperation  Achaa01  Achac01  \
SamplingOperations_code                                                       
S02000008_20170703                                    405      NaN      NaN   
S02000008_20200708                                    400      NaN      NaN   
S02000010_20070906                                    400      NaN      NaN   
S02000010_20090721                                    400      NaN      NaN   
S02000010_20110723                                    398      NaN      NaN   
...                                                   ...      ...      ...   
S06940940_20100708                                    438      NaN      NaN   
S06940940_20230623                                    408      NaN      NaN   
S06960950_20160629                                    401      NaN      NaN   
S06960950_20180719                                    416      NaN      NaN   
S06970900_20150601                                    432      NaN      NaN   

                         Achaf01    Achaf02  Achal01  Acham01  Achan01  \
SamplingOperations_code                                                  
S02000008_20170703           NaN        NaN      NaN      NaN      NaN   
S02000008_20200708           NaN        NaN      NaN      NaN      NaN   
S02000010_20070906           NaN        NaN      NaN      NaN      NaN   
S02000010_20090721           NaN        NaN      NaN      NaN      NaN   
S02000010_20110723           NaN        NaN      NaN      NaN      NaN   
...                          ...        ...      ...      ...      ...   
S06940940_20100708           NaN        NaN      NaN      NaN      NaN   
S06940940_20230623           NaN        NaN      NaN      NaN      NaN   
S06960950_20160629           NaN        NaN      NaN      NaN      NaN   
S06960950_20180719           NaN  19.230769      NaN      NaN      NaN   
S06970900_20150601           NaN        NaN      NaN      NaN      NaN   

                         Achat01   Achat02  Achat03  Achba01  Achbi01  \
SamplingOperations_code                                                 
S02000008_20170703           NaN       NaN      NaN      NaN      NaN   
S02000008_20200708           NaN       NaN      NaN      NaN      NaN   
S02000010_20070906           NaN       NaN      NaN      NaN      NaN   
S02000010_20090721           NaN       NaN      NaN      NaN      NaN   
S02000010_20110723           NaN       NaN      NaN      NaN      NaN   
...                          ...       ...      ...      ...      ...   
S06940940_20100708           NaN       NaN      NaN      NaN      NaN   
S06940940_20230623           NaN       NaN      NaN      NaN      NaN   
S06960950_20160629           NaN       NaN      NaN      NaN      NaN   
S06960950_20180719           NaN       NaN      NaN      NaN      NaN   
S06970900_20150601           NaN  6.944444      NaN      NaN      NaN   

                         Achbi02   Achbr01  Achca01  Achca02  Achca03  \
SamplingOperations_code                                                 
S02000008_20170703           NaN       NaN      NaN      NaN      NaN   
S02000008_20200708           NaN       NaN      NaN      NaN      NaN   
S02000010_20070906           NaN       NaN      NaN      NaN      NaN   
S02000010_20090721           NaN       NaN      NaN      NaN      NaN   
S02000010_20110723           NaN       NaN      NaN      NaN      NaN   
...                          ...       ...      ...      ...      ...   
S06940940_20100708           NaN  2.283105      NaN      NaN      NaN   
S06940940_20230623           NaN       NaN      NaN      NaN      NaN   
S06960950_20160629           NaN       NaN      NaN      NaN      NaN   
S06960950_20180719           NaN       NaN      NaN      NaN      NaN   
S06970900_20150601           NaN       NaN      NaN      NaN      NaN   

                         Achca04  Achch01  Achcl01  Achco01  Achco02  Achco03  \
SamplingOperations_code                              

In [12]:
# X is where IBD is not null
X_train = df[df['IBD'].notnull()].drop(columns=['IBD_EQR'])
y_train = X_train.pop('IBD')

X_test = df[df['IBD'].isnull()].drop(columns=['IBD_EQR'])
y_test = X_test.pop('IBD')


In [ ]:
# =============================== SETUP ========================================
# If needed:
# %pip install -U numpy pandas scikit-learn matplotlib scipy xgboost

import os, math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.model_selection import RepeatedKFold
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import xgboost as xgb
from xgboost import XGBRegressor

# ---------- INPUTS ----------
# Provide your own data objects or load from CSVs.
# X_train = pd.read_csv("X_train.csv")
# y_train = pd.read_csv("y_train.csv").squeeze("columns")
# X_test  = pd.read_csv("X_unlabeled_test.csv")

RANDOM_STATE = 42
N_JOBS = -1
OUTDIR = "model_outputs_gpu_xgb"
os.makedirs(OUTDIR, exist_ok=True)

# Basic checks
assert isinstance(X_train, pd.DataFrame)
assert isinstance(y_train, (pd.Series, np.ndarray))
assert len(X_train) == len(y_train)
assert isinstance(X_test, pd.DataFrame)

# Ensure numeric target
y_train = pd.Series(np.asarray(y_train, dtype=float), index=X_train.index)

# ======================== FEATURE ENGINEERING =================================
# XGBoost handles NaN in numeric features natively. Categoricals need OHE.

num_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = X_train.columns.difference(num_cols).tolist()

pre = ColumnTransformer(
    transformers=[
        ("num", "passthrough", num_cols),
        ("cat",
         Pipeline(steps=[
             ("imp", SimpleImputer(strategy="constant", fill_value="__MISSING__")),
             ("ohe", OneHotEncoder(handle_unknown="ignore", sparse=True))
         ]),
         cat_cols),
    ],
    remainder="drop",
    sparse_threshold=1.0
)

def get_feature_names(preprocessor: ColumnTransformer) -> np.ndarray:
    names = []
    for name, trans, cols in preprocessor.transformers_:
        if name == "remainder":
            continue
        if trans == "passthrough":
            names.extend(list(cols))
        else:
            base = cols
            try:
                gn = trans.get_feature_names_out(base)
            except Exception:
                last = trans.steps[-1][1]
                gn = last.get_feature_names_out(base)
            names.extend(gn)
    return np.array(names, dtype=object)

# =========================== MODEL ============================================
def make_gpu_xgb(n_estimators=5000, lr=0.03, max_depth=8, subsample=0.8, colsample=0.8, reg_lambda=1.0):
    return XGBRegressor(
        tree_method="gpu_hist",
        predictor="gpu_predictor",
        n_estimators=n_estimators,
        learning_rate=lr,
        max_depth=max_depth,
        subsample=subsample,
        colsample_bytree=colsample,
        reg_lambda=reg_lambda,
        random_state=RANDOM_STATE,
        n_jobs=N_JOBS,
        verbosity=0,
        eval_metric="mae"
    )

# ====================== OOF PREDICTIONS + METRICS =============================
rkf = RepeatedKFold(n_splits=5, n_repeats=2, random_state=RANDOM_STATE)

y_oof = np.empty(len(X_train), dtype=float); y_oof[:] = np.nan
best_iters = []

for fold, (tr_idx, va_idx) in enumerate(rkf.split(X_train), 1):
    Xtr, Xva = X_train.iloc[tr_idx], X_train.iloc[va_idx]
    ytr, yva = y_train.iloc[tr_idx], y_train.iloc[va_idx]

    # Fit preprocessor on train of the fold
    pre_fold = pre.fit(Xtr)
    Xtr_t = pre_fold.transform(Xtr)
    Xva_t = pre_fold.transform(Xva)

    # Fit GPU XGBoost with early stopping callback (minimal patch A)
    model = make_gpu_xgb(n_estimators=5000)
    model.fit(
        Xtr_t, ytr,
        eval_set=[(Xva_t, yva)],
        callbacks=[xgb.callback.EarlyStopping(rounds=200, save_best=True)],
        verbose=False
    )

    # Booster is rolled back to best iteration; no iteration_range needed
    yhat = model.predict(Xva_t)
    y_oof[va_idx] = yhat
    best_iters.append(model.best_iteration)

# Metrics
def regression_metrics(y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = math.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    medae = np.median(np.abs(y_true - y_pred))
    eps = np.finfo(float).eps
    mask = np.abs(y_true) > eps
    mape = np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100 if mask.any() else np.nan
    rho, _ = stats.spearmanr(y_true, y_pred)
    return dict(MAE=mae, RMSE=rmse, R2=r2, MedAE=medae, MAPE_pct=mape, Spearman=rho)

metrics_oof = regression_metrics(y_train.values, y_oof)
print("=== Cross-validated (OOF) metrics on 60k ===")
for k, v in metrics_oof.items():
    print(f"{k:>10}: {v:.6f}")
pd.DataFrame({"y_true": y_train, "y_oof": y_oof}).to_csv(f"{OUTDIR}/oof_predictions.csv", index=False)

# ======================= CONFORMAL INTERVALS ==================================
resid = y_train.values - y_oof
def conformal_q(residuals, alpha):
    return np.quantile(np.abs(residuals), 1 - alpha, method="higher")
q90 = conformal_q(resid, 0.10)
q95 = conformal_q(resid, 0.05)
print(f"\nConformal absolute residual quantiles: q90={q90:.6f}, q95={q95:.6f}")

# =================== FINAL FIT ON ALL 60k + PREDICT 6k ========================
pre_all = pre.fit(X_train)
Xt_all  = pre_all.transform(X_train)
Xt_test = pre_all.transform(X_test)
feat_names = get_feature_names(pre_all)

best_n = int(np.ceil(1.10 * np.mean(best_iters))) if len(best_iters) else 2000
final_model = make_gpu_xgb(n_estimators=best_n)
final_model.fit(Xt_all, y_train.values, verbose=False)

y_test_pred = final_model.predict(Xt_test)
test_df = pd.DataFrame({
    "prediction": y_test_pred,
    "pi90_lo": y_test_pred - q90,
    "pi90_hi": y_test_pred + q90,
    "pi95_lo": y_test_pred - q95,
    "pi95_hi": y_test_pred + q95
}, index=X_test.index)
test_df.to_csv(f"{OUTDIR}/test_predictions_with_PIs.csv")
print(f"\nSaved test predictions with 90%/95% PIs → {OUTDIR}/test_predictions_with_PIs.csv")

# ============================= FEATURE IMPORTANCE =============================
gain = final_model.get_booster().get_score(importance_type="gain")
cover = final_model.get_booster().get_score(importance_type="cover")
weight = final_model.get_booster().get_score(importance_type="weight")

def map_importance(imp_dict, names):
    rows = []
    for k, v in imp_dict.items():
        idx = int(k[1:])  # 'f123' -> 123
        rows.append((names[idx] if idx < len(names) else k, v))
    df = pd.DataFrame(rows, columns=["feature", "value"]).sort_values("value", ascending=False)
    return df

imp_gain  = map_importance(gain,  feat_names)
imp_cover = map_importance(cover, feat_names)
imp_weight= map_importance(weight,feat_names)
top_gain = imp_gain.head(25)
top_gain.to_csv(f"{OUTDIR}/top25_importance_gain.csv", index=False)

# ================================ PLOTS =======================================
# Parity plot (OOF)
plt.figure(figsize=(6,6))
lim_lo = np.nanpercentile(np.concatenate([y_train.values, y_oof]), 1)
lim_hi = np.nanpercentile(np.concatenate([y_train.values, y_oof]), 99)
plt.scatter(y_train, y_oof, s=6, alpha=0.4)
plt.plot([lim_lo, lim_hi], [lim_lo, lim_hi], lw=2)
plt.xlabel("Actual (train)")
plt.ylabel("OOF prediction")
plt.title("Parity plot (OOF, XGBoost GPU)")
plt.tight_layout()
plt.savefig(f"{OUTDIR}/plot_parity_oof.png", dpi=160)

# Residuals vs fitted (OOF)
plt.figure(figsize=(7,4))
plt.scatter(y_oof, resid, s=6, alpha=0.4)
plt.axhline(0, lw=1)
plt.xlabel("OOF prediction")
plt.ylabel("Residual")
plt.title("Residuals vs fitted (OOF)")
plt.tight_layout()
plt.savefig(f"{OUTDIR}/plot_resid_vs_fit.png", dpi=160)

# Residual histogram
plt.figure(figsize=(7,4))
plt.hist(resid, bins=60, alpha=0.85)
plt.xlabel("Residual")
plt.ylabel("Count")
plt.title("Residual histogram (OOF)")
plt.tight_layout()
plt.savefig(f"{OUTDIR}/plot_resid_hist.png", dpi=160)

# QQ plot
plt.figure(figsize=(6,6))
stats.probplot(resid, dist="norm", plot=plt)
plt.title("QQ plot of residuals (OOF)")
plt.tight_layout()
plt.savefig(f"{OUTDIR}/plot_resid_qq.png", dpi=160)

# Top 25 by gain
plt.figure(figsize=(8,8))
plt.barh(top_gain["feature"][::-1], top_gain["value"][::-1])
plt.xlabel("Gain")
plt.title("Top 25 features by gain (XGBoost GPU)")
plt.tight_layout()
plt.savefig(f"{OUTDIR}/plot_top25_gain.png", dpi=160)

print("\nArtifacts saved in", OUTDIR)
print("OOF metrics:", metrics_oof)
print("Avg best_iteration across folds:", int(np.mean(best_iters)))


AttributeError: `best_iteration` is only defined when early stopping is used.